# SVD — Cross-Check & Diagnostics (Jenny)

Loads Kate's **final tuned** SVD (`svd_model.pkl`, n_factors=200) — the single SVD model for the
project — and uses it to (1) **validate the shared evaluation pipeline** against
TwoTower/Popularity and (2) **diagnose the RMSE–NDCG divergence**. No model is trained here:
all SVD training lives in `model.ipynb`. The default-param SVD appears only as a one-line
comparison baseline (numbers cited from `model.ipynb`).

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from surprise import accuracy

RANDOM_STATE = 42

## 2. Load Data & Build Evaluation Structures

Load the shared train/test splits and build the per-user structures the evaluation needs:
`train_items_by_user` (to exclude seen items when sampling negatives) and
`test_ratings_by_user` ({user: {book: rating}}, the graded-relevance source).

In [2]:
train = pd.read_csv("train_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

# Per-user train items (exclude when sampling negatives) and test ratings (graded relevance)
train_items_by_user = train.groupby("user_id")["book_id"].apply(set).to_dict()
test_ratings_by_user = (
    test.groupby("user_id")
        .apply(lambda g: dict(zip(g["book_id"], g["rating"])), include_groups=False)
        .to_dict()
)
all_items = train["book_id"].unique()

print(f"Users with test data: {len(test_ratings_by_user):,}")

Users with test data: 58,787


## 3. Shared Evaluation Pipeline

`build_candidate_pools` and `evaluate_ndcg` are **identical** to those in `twotower.ipynb`
(same seed=42, n_neg=100), so all models are scored on identical candidate sets — this is
what makes the cross-model comparison fair.

In [3]:
def build_candidate_pools(test_ratings_by_user, train_items_by_user,
                          all_items, n_neg=100, seed=RANDOM_STATE):
    """Per user: pool = their test items (positives) + n_neg sampled negatives
    (excluding anything they interacted with in train). Deterministic via seed."""
    rng = np.random.default_rng(seed)
    all_items_arr = np.asarray(all_items)
    pools = {}
    for user, test_items in test_ratings_by_user.items():
        seen = train_items_by_user.get(user, set())
        pos_items = set(test_items.keys())
        exclude = seen | pos_items
        negs = []
        while len(negs) < n_neg:
            cand = rng.choice(all_items_arr, size=n_neg * 2, replace=False)
            negs = [it for it in cand if it not in exclude][:n_neg]
        pools[user] = list(pos_items) + negs
    return pools

candidate_pools = build_candidate_pools(
    test_ratings_by_user, train_items_by_user, all_items, n_neg=100
)
print(f"Built candidate pools for {len(candidate_pools):,} users")

Built candidate pools for 58,787 users


In [4]:
def evaluate_ndcg(score_fn, candidate_pools, test_ratings_by_user, k=10):
    """Graded NDCG@k. y_true = true test rating (else 0); y_score = model's scores.
    Only score_fn varies across models; pools/ground-truth/metric are shared."""
    ndcgs = []
    for user, items in candidate_pools.items():
        ratings = test_ratings_by_user[user]
        y_true = [ratings.get(it, 0) for it in items]
        if sum(y_true) == 0 or len(set(y_true)) == 1:   # need >=1 positive and some variation
            continue
        y_score = score_fn(user, items)
        ndcgs.append(ndcg_score([y_true], [y_score], k=k))
    return float(np.mean(ndcgs)), len(ndcgs)

## 4. Load the Final Tuned Model

Load the tuned SVD saved by `model.ipynb` (`svd_model.pkl`: n_factors=200, n_epochs=40,
lr=0.02, reg=0.1). This is the single SVD model used everywhere below — RMSE (§5), NDCG (§6),
the ranking-ization experiment (§7), and the diagnostics (§8) all run on it. We do **not**
retrain here; the default-param SVD is referenced only as a comparison baseline.

In [5]:
import pickle

with open("svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)          # Kate's final tuned SVD (nf=200, ep=40, lr=0.02, reg=0.1)

# The fitted model carries its own trainset (raw<->inner id maps + learned pu/qi/biases),
# so the latent-vector lookups in §7 stay consistent with the saved factors.
trainset = svd_model.trainset
print(f"Loaded tuned SVD: n_factors={svd_model.n_factors}, n_epochs={svd_model.n_epochs} "
      f"| users={trainset.n_users:,}, items={trainset.n_items:,}")

Loaded tuned SVD: n_factors=200, n_epochs=40 | users=61,078, items=22,931


## 5. Rating Accuracy — RMSE vs a Naive Baseline

RMSE is SVD's **native** rating-prediction metric (NOT a cross-model comparison — TwoTower has
no RMSE). To judge whether the tuned SVD's RMSE reflects real learning, we compare it to a
**global-mean baseline** (predict the train mean for every test rating), whose RMSE equals the
rating std. The default-param SVD's RMSE (from `model.ipynb`) is shown as a reference — tuning
barely moves it.

In [6]:
# Global-mean baseline: predict the train mean for every test rating.
# Answers "does SVD learn signal beyond a no-information predictor?"
global_mean = train["rating"].mean()
baseline_rmse = np.sqrt(np.mean((test["rating"] - global_mean) ** 2))

# Tuned SVD's own RMSE on the test set (rating-prediction task; no candidate pools)
testset = list(test[["user_id", "book_id", "rating"]].itertuples(index=False, name=None))
svd_rmse = accuracy.rmse(svd_model.test(testset), verbose=False)
svd_mae  = accuracy.mae(svd_model.test(testset), verbose=False)

print(f"Global-mean baseline test RMSE: {baseline_rmse:.4f}")
print(f"SVD (tuned) test RMSE:          {svd_rmse:.4f}")
print(f"SVD (tuned) test MAE:           {svd_mae:.4f}")
print(f"Improvement over naive baseline: {(baseline_rmse - svd_rmse)/baseline_rmse*100:.1f}%")
print(f"Reference — SVD (default, model.ipynb) test RMSE: 0.7614  (tuning Δ ≈ noise)")

Global-mean baseline test RMSE: 0.9388
SVD (tuned) test RMSE:          0.7627
SVD (tuned) test MAE:           0.5879
Improvement over naive baseline: 18.8%
Reference — SVD (default, model.ipynb) test RMSE: 0.7614  (tuning Δ ≈ noise)


## 6. Graded NDCG — SVD as a Rating Model

Score candidates by the tuned SVD's predicted rating (`.est`) and rank. This is the cross-model
number, on the SAME pools as TwoTower/Popularity.

In [7]:
def svd_score_fn(user, items):
    return [svd_model.predict(user, it).est for it in items]

svd_ndcg, n_users = evaluate_ndcg(svd_score_fn, candidate_pools, test_ratings_by_user, k=10)
print(f"SVD (tuned, as rating est) — graded NDCG@10: {svd_ndcg:.4f}  (over {n_users:,} users)")
print(f"Reference: SVD default 0.1587 | Popularity 0.6992 | TwoTower 0.8508 (same pools)")

SVD (tuned, as rating est) — graded NDCG@10: 0.1628  (over 58,787 users)
Reference: SVD default 0.1587 | Popularity 0.6992 | TwoTower 0.8508 (same pools)


## 7. Graded NDCG — SVD as a Ranking Model (ranking-ization)

Re-score with the raw latent-vector dot product `pu·qi` (no biases/global mean) — the direct
analogue of TwoTower's similarity. Controls the "usage" variable to isolate **architecture**.

In [8]:
def svd_ranking_score_fn(user, items):
    """SVD as a ranking model: score = user factor . item factor (raw pu/qi dot,
    no biases) — closest analogue to TwoTower's latent-vector similarity."""
    inner_u = trainset.to_inner_uid(user)
    u_vec = svd_model.pu[inner_u]
    scores = []
    for it in items:
        try:
            inner_i = trainset.to_inner_iid(it)
            scores.append(float(np.dot(u_vec, svd_model.qi[inner_i])))
        except ValueError:
            scores.append(-np.inf)        # item unseen in train -> lowest
    return scores

svd_rank_ndcg, n_users = evaluate_ndcg(
    svd_ranking_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"SVD (as ranking, pu.qi) — graded NDCG@10: {svd_rank_ndcg:.4f}  (over {n_users:,} users)")
print(f"Compare: SVD-as-rating {svd_ndcg:.4f} | Popularity 0.6992 | TwoTower 0.8508")

SVD (as ranking, pu.qi) — graded NDCG@10: 0.2242  (over 58,787 users)
Compare: SVD-as-rating 0.1628 | Popularity 0.6992 | TwoTower 0.8508


## 8. Diagnostic — Why SVD Ranks Poorly (RMSE–NDCG divergence)

SVD has good RMSE but poor NDCG. These diagnostics show why: its predicted ratings cluster near
the global mean, so positives and negatives are barely separated.

In [9]:
# 8a. What does SVD predict for one user's pool? (sanity check)
u0 = next(iter(candidate_pools))
items0 = candidate_pools[u0]
preds = [svd_model.predict(u0, it) for it in items0[:10]]
for p in preds:
    print(f"item={p.iid}  est={p.est:.4f}  impossible={p.details.get('was_impossible')}")

item=6250208  est=3.5169  impossible=False
item=1258121  est=3.5787  impossible=False
item=297676  est=3.4436  impossible=False
item=3636  est=4.1429  impossible=False
item=11788115  est=3.5385  impossible=False
item=14823888  est=3.5414  impossible=False
item=982432  est=3.9119  impossible=False
item=20307024  est=4.0363  impossible=False
item=23600172  est=3.4760  impossible=False
item=92637  est=3.9380  impossible=False


In [10]:
# 8b. Are positives scored higher than negatives for one user? By how much?
u0 = next(iter(candidate_pools))
items0 = candidate_pools[u0]
ratings0 = test_ratings_by_user[u0]

pos_est, neg_est = [], []
for it in items0:
    est = svd_model.predict(u0, it).est
    (pos_est if it in ratings0 else neg_est).append(est)

print(f"User {u0[:8]}")
print(f"  positives: {len(pos_est)} items, est mean={np.mean(pos_est):.3f}, range=[{min(pos_est):.2f}, {max(pos_est):.2f}]")
print(f"  negatives: {len(neg_est)} items, est mean={np.mean(neg_est):.3f}, range=[{min(neg_est):.2f}, {max(neg_est):.2f}]")
print(f"  -> positives scored higher? {np.mean(pos_est) > np.mean(neg_est)}")

User 00004584
  positives: 4 items, est mean=3.671, range=[3.44, 4.14]
  negatives: 100 items, est mean=3.745, range=[3.07, 4.52]
  -> positives scored higher? False


In [11]:
# 8c. Is the pos/neg gap small across MANY users (not just one)?
gaps = []
for user, items in list(candidate_pools.items())[:2000]:
    ratings = test_ratings_by_user[user]
    pos, neg = [], []
    for it in items:
        est = svd_model.predict(user, it).est
        (pos if it in ratings else neg).append(est)
    if pos and neg:
        gaps.append(np.mean(pos) - np.mean(neg))

gaps = np.array(gaps)
print(f"Over {len(gaps)} users:")
print(f"  mean (pos_est - neg_est): {gaps.mean():.4f}")
print(f"  fraction where positives scored higher: {(gaps > 0).mean():.1%}")
print("Interpretation: right direction (>50%) but weak separation (~0.1 on a 1-5 scale)")
print("-> good RMSE, poor ranking (the RMSE-NDCG divergence).")

Over 2000 users:
  mean (pos_est - neg_est): 0.1447
  fraction where positives scored higher: 85.0%
Interpretation: right direction (>50%) but weak separation (~0.1 on a 1-5 scale)
-> good RMSE, poor ranking (the RMSE-NDCG divergence).
